In [1]:
print("hello world")

hello world


# Basic LLM call with JSON structure

In [2]:
import os
import json
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
model_name = "openai/gpt-oss-120b"
client = Groq(api_key=api_key)

response = client.chat.completions.create(
    model=model_name,
    messages=[
        {
            "role": "system",
            "content": "You are a CTF-agent designed to autonomously navigate environments and perform standard CTF tasks and find flags."
        },
        {
            "role": "user",
            "content": (
                "you are dropped in a new environment."
                "Reason first and then provide the next logical shell command. "
                "Respond with JSON: {\"reasoning\": \"your_reasoning_here\", \"shell_command\": \"your_command_here\"}"
            )
        }
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "shell_command",
            "schema": {
                "type": "object",
                "properties": {
                    "reasoning": {"type": "string"},
                    "shell_command": {"type": "string"}
                },
                "required": ["reasoning", "shell_command"],
                "additionalProperties": False
            }
        }
    }
)

result = json.loads(response.choices[0].message.content)
print(json.dumps(result, indent=2))

{
  "reasoning": "To understand the new environment we should identify the current user, OS, privileges, working directory, and visible files. This information guides further enumeration and exploitation steps.",
  "shell_command": "whoami; uname -a; id; pwd; ls -la"
}


# Docker Linux VM test with a shell command

In [ ]:
import docker

client = docker.from_env()
container = client.containers.get("kali-linux")  # OBS! ensure it's running (e.g., via docker compose up -d)
exit_code, output = container.exec_run(
    ["bash", "-lc", "ls -la"], # "ls -la is the comamand"
)


print(output.decode().strip())
assert exit_code == 0


total 8
drwxr-xr-x 2 root root 4096 Sep 23 12:30 .
drwxr-xr-x 1 root root 4096 Sep 27 08:45 ..


/Users/mariusvangeli/Library/CloudStorage/OneDrive-KTH/work/Hackathons/ctf-agent/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
shell_command = result["shell_command"]
print(shell_command)

whoami; uname -a; id; pwd; ls -la


In [5]:
import docker

client = docker.from_env()
container = client.containers.get("kali-linux")  # ensure it's running (e.g., via docker compose up -d)
exit_code, output = container.exec_run(
    ["bash", "-lc", shell_command],
)


print(output.decode().strip())
assert exit_code == 0


root
Linux 7bfc0cf9aa1f 6.10.14-linuxkit #1 SMP Wed Sep  3 15:35:15 UTC 2025 aarch64 GNU/Linux
uid=0(root) gid=0(root) groups=0(root)
/app
total 8
drwxr-xr-x 2 root root 4096 Sep 23 12:30 .
drwxr-xr-x 1 root root 4096 Sep 27 08:45 ..
